# Step 12 – Comparison with Hansen Global Forest Change: agreement map

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Section 3.7
- **Needs:** CatBoost map tiles (`DATA_DIR/maps/CatBoost/`) and `DATA_DIR/P2_hansen_10m.tif`
- **Writes:** `RESULTS_DIR/hansen_comparison/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / P2_hansen_10m.tif
#   DATA_DIR / maps/CatBoost/
#   RESULTS_DIR / hansen_comparison/

In [ ]:
!pip install -q rasterio

In [ ]:
"""
Agreement between the CatBoost forest map and Hansen GFC

Run after the CatBoost map (DATA_DIR/maps/CatBoost/) and P2_hansen_10m.tif exist
(can run in the same session as steps 11 and 13).   In Colab first:  !pip install -q rasterio

For every 10 m pixel on the common grid:
  1 = forest in both   2 = CatBoost only   3 = Hansen only   4 = non-forest in both
Hansen forest = tree cover 2000 >= THRESHOLD %, minus loss 2001-(2000+LOSS_UNTIL),
exactly as in the threshold comparison (step 11), so the areas match that table.

Outputs (OUT_DIR):
  agreement_10m/*.tif                 agreement classes at 10 m (for the data archive)
  Fig10_catboost_hansen_agreement.(png|pdf)   (a) study area, (b) detail, with legend
  fig10_agreement_areas.csv           area of each agreement class (km2, %)
"""


import os, glob
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch, Rectangle
from rasterio.windows import from_bounds
from rasterio.merge import merge
from rasterio.enums import Resampling
from scipy.ndimage import uniform_filter

# ------------------------------------------------------------------ config
MAP_DIR     = DATA_DIR + 'maps/CatBoost/'
HANSEN_PATH = DATA_DIR + 'P2_hansen_10m.tif'
OUT_DIR     = RESULTS_DIR + 'hansen_comparison/'
TILE_DIR    = OUT_DIR + 'agreement_10m/'
FOREST_CODE = 1
NODATA      = 255
THRESHOLD   = 30           # Hansen tree cover (%), as in Table 5
LOSS_UNTIL  = 24           # remove Hansen loss 2001..2024
DISPLAY_RES = 30           # m, panel (a)
INSET_BOUNDS = None        # (left, bottom, right, top) in EPSG:32629 m; None = automatic
INSET_SIZE_KM = 10
CLASSES = {1: ('Forest in both', '#1B7837'),
           2: ('CatBoost forest only', '#A6DBA0'),
           3: ('Hansen forest only', '#D95F02'),
           4: ('Non-forest in both', '#E6E6E6')}

os.makedirs(TILE_DIR, exist_ok=True)
tiles = sorted(glob.glob(MAP_DIR + '*.tif'))
assert tiles, f'No CatBoost map tiles in {MAP_DIR}'
hansen = rasterio.open(HANSEN_PATH)

# ------------------------------------------------------------------ 10 m agreement tiles
counts = np.zeros(256, dtype=np.int64)
out_tiles = []
for t in tiles:
    with rasterio.open(t) as src:
        cb, prof, bounds = src.read(1), src.profile.copy(), src.bounds
    win = from_bounds(*bounds, transform=hansen.transform)
    assert np.allclose([win.col_off, win.row_off], np.round([win.col_off, win.row_off]), atol=1e-3), \
        'CatBoost and Hansen pixels are not aligned'
    win = win.round_offsets().round_lengths()
    tc, loss, gain, dmask = hansen.read(window=win, boundless=True, fill_value=NODATA)[:, :cb.shape[0], :cb.shape[1]]
    valid = (cb != NODATA) & (tc != NODATA) & (dmask == 1)
    c_for = cb == FOREST_CODE
    h_for = tc >= THRESHOLD
    if LOSS_UNTIL:
        h_for &= ~((loss >= 1) & (loss <= LOSS_UNTIL))
    a = np.full(cb.shape, NODATA, dtype=np.uint8)
    a[valid & c_for & h_for] = 1
    a[valid & c_for & ~h_for] = 2
    a[valid & ~c_for & h_for] = 3
    a[valid & ~c_for & ~h_for] = 4
    counts += np.bincount(a.ravel(), minlength=256)
    prof.update(dtype='uint8', nodata=NODATA, count=1, compress='deflate')
    out = TILE_DIR + os.path.basename(t).replace('_CatBoost.tif', '_agreement.tif')
    with rasterio.open(out, 'w', **prof) as dst:
        dst.write(a, 1)
    out_tiles.append(out)
    print(os.path.basename(out), 'done')

px_km2 = abs(hansen.transform.a * hansen.transform.e) / 1e6
area = pd.DataFrame({'Agreement class': [CLASSES[k][0] for k in CLASSES],
                     'Area (km2)': [counts[k] * px_km2 for k in CLASSES]})
area['Share (%)'] = 100 * area['Area (km2)'] / area['Area (km2)'].sum()
area = area.round(2)
area.to_csv(OUT_DIR + 'fig10_agreement_areas.csv', index=False)
print(f'\n=== Agreement areas (Hansen >= {THRESHOLD}%, loss 2001-{2000 + LOSS_UNTIL} removed) ===')
print(area.to_string(index=False))

# ------------------------------------------------------------------ mosaics
cmap = ListedColormap([CLASSES[k][1] for k in CLASSES])
norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)
full, tr = merge(out_tiles, res=DISPLAY_RES, nodata=NODATA, resampling=Resampling.mode)
full = full[0]
h, w = full.shape
ext = [tr.c / 1000, (tr.c + w * tr.a) / 1000, (tr.f + h * tr.e) / 1000, tr.f / 1000]

if INSET_BOUNDS is None:        # window with the most Hansen forest (both + Hansen-only)
    win_px = int(INSET_SIZE_KM * 1000 / DISPLAY_RES)
    half = win_px // 2
    hf = np.isin(full, [1, 3]).astype(float)
    ok = uniform_filter((full != NODATA).astype(float), win_px, mode='constant') > 0.98
    score = uniform_filter(hf, win_px, mode='constant') * ok
    score[:half + 1, :] = 0; score[-half - 1:, :] = 0; score[:, :half + 1] = 0; score[:, -half - 1:] = 0
    r, c = np.unravel_index(np.argmax(score), score.shape)
    x0, y1 = tr.c + (c - half) * tr.a, tr.f + (r - half) * tr.e
    INSET_BOUNDS = (x0, y1 - win_px * DISPLAY_RES, x0 + win_px * DISPLAY_RES, y1)
    print('\nAuto-selected detail window:', [round(v) for v in INSET_BOUNDS],
          '- set INSET_BOUNDS by hand to show another area')
l, b, rr, t = INSET_BOUNDS
det, _ = merge(out_tiles, res=10, bounds=INSET_BOUNDS, nodata=NODATA, resampling=Resampling.nearest)
det = det[0]

# ------------------------------------------------------------------ figure
def scalebar(ax, km, e):
    x0 = e[0] + 0.06 * (e[1] - e[0]); y0 = e[2] + 0.05 * (e[3] - e[2])
    ax.plot([x0, x0 + km], [y0, y0], color='black', lw=2)
    ax.text(x0 + km / 2, y0 + 0.02 * (e[3] - e[2]), f'{km:g} km', ha='center', va='bottom', fontsize=7,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1))

def north(ax):
    ax.annotate('N', xy=(0.08, 0.95), xytext=(0.08, 0.83), xycoords='axes fraction', ha='center',
                fontsize=8, fontweight='bold', arrowprops=dict(arrowstyle='-|>', color='black', lw=1.2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.1, 4.4), gridspec_kw={'width_ratios': [1.25, 1]})
ax1.imshow(np.ma.masked_equal(full, NODATA), cmap=cmap, norm=norm, extent=ext, interpolation='nearest')
ax1.add_patch(Rectangle((l / 1000, b / 1000), (rr - l) / 1000, (t - b) / 1000, fill=False, ec='black', lw=1))
ax1.set_title('(a) Study area', fontsize=8.5, fontweight='bold')
scalebar(ax1, max(10, round((ext[1] - ext[0]) / 5, -1)), ext)
north(ax1)
det_ext = [l / 1000, rr / 1000, b / 1000, t / 1000]
ax2.imshow(np.ma.masked_equal(det, NODATA), cmap=cmap, norm=norm, extent=det_ext, interpolation='nearest')
ax2.set_title('(b) Detail (10 m pixels)', fontsize=8.5, fontweight='bold')
scalebar(ax2, max(1, round((rr - l) / 1000 / 4)), det_ext)
for ax in (ax1, ax2):
    ax.set_xticks([]); ax.set_yticks([])
handles = [Patch(fc=CLASSES[k][1], ec='#999999' if k == 4 else 'none', lw=0.5,
                 label=f"{CLASSES[k][0]} ({area.loc[i, 'Area (km2)']:,.0f} km², {area.loc[i, 'Share (%)']:.1f}%)")
           for i, k in enumerate(CLASSES)]
fig.legend(handles=handles, loc='lower center', ncol=2, frameon=False, fontsize=7.5,
           title=f'Hansen forest: tree cover ≥ {THRESHOLD}% (2000), loss 2001–{2000 + LOSS_UNTIL} removed',
           title_fontsize=7.5)
fig.tight_layout(rect=(0, 0.13, 1, 1))
fig.savefig(OUT_DIR + 'Fig10_catboost_hansen_agreement.png', dpi=600, bbox_inches='tight')
fig.savefig(OUT_DIR + 'Fig10_catboost_hansen_agreement.pdf', bbox_inches='tight')
plt.show()
print('\nFiles saved to:', OUT_DIR)